# 🧳 Assignment: AI Travel Adventure Planner

## 🎯 Objective

Build an **AI Travel Adventure Planner** using **LangChain**, **OpenRouter**, and **Pydantic Structured Output**.

The application should collect travel preferences from the user and generate a structured travel itinerary.

---

## 📋 Scenario

The user provides:

- 🌍 Destination
- 📅 Number of Days
- 🎒 Travel Style
- 💰 Budget Level

The AI generates a travel plan containing:

- 🏷️ Trip Title
- 🌍 Destination
- 📝 Introduction
- 📍 Places to Visit (Exactly 3)
- 🍜 Food to Try
- 💡 Travel Tip
- 💰 Estimated Budget

The response must be returned as a **Pydantic object**, not plain text.

---

# ✅ Requirements

## 1️⃣ API Key Setup

- Store the OpenRouter API key in a `.env` file.
- Load the environment variables.
- Read `OPENROUTER_API_KEY`.
- Check whether the key exists.
- Display an appropriate message.

Example:

```
OpenRouter API Key Found
```

> ❌ Do not hardcode the API key.

---

## 2️⃣ Create the Chat Model

Create a `ChatOpenAI` model using:

- OpenRouter API
- A free GPT model
- Temperature = `0.7`

API Endpoint:

```
https://openrouter.ai/api/v1
```

---

## 3️⃣ Collect User Input

Prompt the user for:

- Destination
- Number of Days
- Travel Style
- Budget Level

Store each value in a separate variable.

---

## 4️⃣ Create a ChatPromptTemplate

Use:

```python
ChatPromptTemplate.from_messages()
```

Include:

- 🛡️ System Message
- 🙋 User Message

The user message should use placeholders:

- `{destination}`
- `{days}`
- `{travel_style}`
- `{budget}`

---

## 5️⃣ Create a Pydantic Model

Create a model named:

```
TravelPlan
```

Fields:

- trip_title
- destination
- introduction
- places_to_visit
- food_to_try
- travel_tip
- estimated_budget

`places_to_visit` must contain exactly **3 places**.

---

## 6️⃣ Add Field Descriptions

Every field must use:

```python
Field(description="...")
```

Descriptions should clearly explain the purpose of each field.

---

## 7️⃣ Structured Output

Convert the LLM into a structured model using:

```
with_structured_output()
```

The output should follow the `TravelPlan` schema.

---

## 8️⃣ Demonstrate Messages

Create a short conversation using:

- 🛡️ SystemMessage
- 🙋 HumanMessage
- 🤖 AIMessage

Then ask one final travel-related question and print the response.

---

## 9️⃣ Generate the Final Travel Plan

Flow:

```
User Input
      ↓
ChatPromptTemplate
      ↓
Structured LLM
      ↓
TravelPlan Object
```

---

## 🔟 Access Individual Fields

Do not print the entire object.

Print each field individually.

For `places_to_visit`, print each location separately.

---

# 🎉 Expected Output

```
========================================
          AI TRAVEL PLANNER
========================================

Trip:
Dubai Luxury Escape

Destination:
Dubai

Introduction:
Experience four days of modern attractions,
desert adventures and world-class experiences.

Places to Visit:

1. Burj Khalifa
2. Dubai Marina
3. Desert Safari

Food to Try:
Machboos

Travel Tip:
Book major attractions in advance.

Estimated Budget:
₹80,000 - ₹1,00,000

========================================
Have a Great Journey!
========================================
```

### 📚 Concepts Covered

- 🌱 Environment Variables (`.env`)
- 🤖 ChatOpenAI
- 💬 ChatPromptTemplate
- 📨 System, Human & AI Messages
- 🏗️ Pydantic Models
- 🏷️ Field Descriptions
- 📦 Structured Output
- 🎯 Accessing Individual Object Fields

In [5]:
import os
from dotenv import load_dotenv


from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)

# =====================================================
# 1. Load API Key
# =====================================================

# load_dotenv()

load_dotenv()

if os.getenv("OPENROUTER_API_KEY") is not None:
    print("OpenRouter API Key Found")
else:
    print("OpenRouter API Key Not Found")

# =====================================================
# 2. Create Chat Model
# =====================================================

llm_openai = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7
)

# =====================================================
# 3. Collect User Input
# =====================================================

destination = input("Enter destination: ")
days = input("Enter number of days: ")
travel_style = input("Enter travel style: ")
budget = input("Enter budget level: ")

# =====================================================
# 4. ChatPromptTemplate
# =====================================================

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert travel planner who creates practical, "
            "interesting and beginner-friendly travel plans."
        ),
        (
            "human",
            """
Create a {days}-day travel plan for {destination}.

Travel Style: {travel_style}

Budget Level: {budget}

Return exactly 3 places to visit.
"""
        ),
    ]
)

# =====================================================
# 5 & 6. Pydantic Model with Field Descriptions
# =====================================================

class TravelPlan(BaseModel):

    trip_title: str = Field(
        description="A creative title for the travel plan."
    )

    destination: str = Field(
        description="The destination chosen by the traveler."
    )

    introduction: str = Field(
        description="A short introduction describing the travel experience."
    )

    places_to_visit: list[str] = Field(
        description="Exactly three recommended places to visit."
    )

    food_to_try: str = Field(
        description="One famous local food the traveler should try."
    )

    travel_tip: str = Field(
        description="One practical travel tip for the trip."
    )

    estimated_budget: str = Field(
        description="Estimated total budget for the trip."
    )

# =====================================================
# 7. Structured Output
# =====================================================

structured_llm = llm_openai.with_structured_output(TravelPlan)

# =====================================================
# 8. Demonstrate Messages
# =====================================================

print("\n==============================")
print("MESSAGE DEMONSTRATION")
print("==============================")

messages = [
    SystemMessage(
        content="You are a friendly travel expert."
    ),
    HumanMessage(
        content="I love adventure trips."
    ),
    AIMessage(
        content="Great! Adventure trips are perfect for exploring exciting places."
    ),
    HumanMessage(
        content="Give me one important travel tip."
    )
]

response = llm_openai.invoke(messages)

print(response.content)

# =====================================================
# 9. Generate Structured Travel Plan
# =====================================================

formatted_messages = prompt.invoke(
    {
        "destination": destination,
        "days": days,
        "travel_style": travel_style,
        "budget": budget
    }
)

travel_plan = structured_llm.invoke(formatted_messages)

# =====================================================
# 10. Access Individual Fields
# =====================================================

print("\n========================================")
print("          AI TRAVEL PLANNER")
print("========================================\n")

print("Trip:")
print(travel_plan.trip_title)

print("\nDestination:")
print(travel_plan.destination)

print("\nIntroduction:")
print(travel_plan.introduction)

print("\nPlaces to Visit:")

for i, place in enumerate(travel_plan.places_to_visit, start=1):
    print(f"{i}. {place}")

print("\nFood to Try:")
print(travel_plan.food_to_try)

print("\nTravel Tip:")
print(travel_plan.travel_tip)

print("\nEstimated Budget:")
print(travel_plan.estimated_budget)

print("\n========================================")
print("Have a Great Journey!")
print("========================================")


response = llm_openai.invoke(formatted_messages)

print(response.content)

OpenRouter API Key Found

MESSAGE DEMONSTRATION
**Plan, but stay flexible.**  
Book your flights, accommodations, and must‑see activities in advance so you won’t waste time scrambling, but keep a few open days (or a flexible itinerary) for spontaneous detours—whether it’s a local festival, a hidden trail, or a last‑minute recommendation from a fellow traveler. Flexibility lets adventure unfold naturally while still keeping the core of your trip on track.


ValidationError: 1 validation error for TravelPlan
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='**3‑Day Train‑Based ...within a modest budget!', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid